# Лабораторная 4 Гипотезы и T тесты

**Иноземцев Сергей Викторович · ЕТ-216 · вариант 11**

Источник задания: вложения «ЛР 4_1 Гипотезы», «ЛР 4_2 Т-тесты», «ЛР 4-Бонус». Все вычисления воспроизводимы при последовательном запуске ячеек. Фиксированные random seed заданы в коде.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'datasets'/'StudentsPerformance.csv').is_file())
DATA = BASE/'datasets'
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
def show(name):
    plt.tight_layout()
    plt.savefig(FIG/(name+'.png'), bbox_inches='tight')
    plt.show()

## Постановка и выбор методов
Выполняются прикреплённые задания «Гипотезы», «Т-тесты» и бонус «Проверка нормальности». Сравниваются независимые группы, поэтому используется двусторонний t-тест Уэлча (equal_var=False), не требующий равенства дисперсий. H0: средние равны; H1: средние различаются; α=0,05. Для серии сравнений дополнительно приведена поправка Holm. Неотклонение H0 не доказывает равенство распределений. Номер учащегося и код профиля не считаются измеряемыми психологическими шкалами.

## Новорождённые Проверка структуры и единиц

In [ ]:
birth=pd.read_excel(DATA/'Пренатальная диагностика.xlsx',sheet_name='исходные данные')
display(birth.head());display(birth.isna().sum());display(birth['pol(1-м, 0-д)'].value_counts().rename(index={0:'девочки',1:'мальчики'}))
assert set(birth['pol(1-м, 0-д)'].unique())<={0,1}
birth['ИМТ']=(birth.ves/1000)/(birth.rost/100)**2
num=[c for c in birth.select_dtypes('number').columns if c!='pol(1-м, 0-д)']
descr=birth[num].describe().T;descr['range']=birth[num].max()-birth[num].min();descr['median']=birth[num].median();display(descr)

ИМТ рассчитан как масса в килограммах / рост в метрах²; исходные единицы — граммы и сантиметры. Здесь показатель используется только как учебный числовой признак, без медицинских заключений. Код пола бинарный и не проверяется на непрерывное нормальное распределение.

In [ ]:
normal=[]
fig,axes=plt.subplots(3,3,figsize=(15,12))
for col,ax in zip(num,axes.flat):
    x=birth[col].dropna();sns.histplot(x,kde=True,ax=ax);ax.set_title(col)
    test=stats.shapiro(x);normal.append([col,len(x),x.mean(),x.median(),stats.skew(x,bias=False),stats.kurtosis(x,bias=False),test.statistic,test.pvalue])
for ax in axes.flat[len(num):]:ax.axis('off')
show('birth_normality')
normal=pd.DataFrame(normal,columns=['показатель','n','mean','median','skew','excess','W','p']);normal['H0 нормальность отклонена']=normal.p<.05;display(normal)

При нормальном распределении теоретические асимметрия и эксцесс равны 0, среднее и медиана совпадают. Выборочные оценки не обязаны быть точно нулевыми. Shapiro проверяет H0 о нормальности; при большом n он выявляет и небольшие отклонения. Визуализация и величина отклонений нужны вместе с p-value.

In [ ]:
def holm(pvalues):
    p=np.asarray(pvalues);order=np.argsort(p);adjusted=np.empty(len(p))
    adjusted[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]));return adjusted
def compare_groups(frame,group,cols,a,b):
    rows=[]
    for col in cols:
        x=frame.loc[frame[group].eq(a),col].dropna().to_numpy();y=frame.loc[frame[group].eq(b),col].dropna().to_numpy()
        t,p=stats.ttest_ind(x,y,equal_var=False)
        diff=x.mean()-y.mean();vx=x.var(ddof=1)/len(x);vy=y.var(ddof=1)/len(y);se=np.sqrt(vx+vy)
        dof=(vx+vy)**2/(vx*vx/(len(x)-1)+vy*vy/(len(y)-1));margin=stats.t.ppf(.975,dof)*se
        pooled=np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/(len(x)+len(y)-2))
        rows.append([col,len(x),len(y),x.mean(),y.mean(),diff,t,p,diff-margin,diff+margin,diff/pooled])
    r=pd.DataFrame(rows,columns=['показатель','n_A','n_B','mean_A','mean_B','A-B','t','p','CI_low','CI_high','Cohen_d'])
    r['p_Holm']=holm(r.p);r['вывод_0.05']=np.where(r.p_Holm<.05,'H0 отклонена','H0 не отклонена');return r
birth_tests=compare_groups(birth,'pol(1-м, 0-д)',['rost','ves','ИМТ'],1,0);display(birth_tests)
birth_tests.to_csv('birth_welch.csv',index=False)

Сравнение мальчики минус девочки:

**rost**: разность 0.674, 95% ДИ [0.424; 0.924], p=1.442e-07, p Holm=2.883e-07; есть статистические основания отвергнуть равенство средних.

**ves**: разность 136.915, 95% ДИ [89.600; 184.229], p=1.765e-08, p Holm=5.294e-08; есть статистические основания отвергнуть равенство средних.

**ИМТ**: разность 0.169, 95% ДИ [0.046; 0.292], p=0.006948, p Holm=0.006948; есть статистические основания отвергнуть равенство средних.

Группы независимы, дисперсии не приравниваются. Большой объём смягчает ненормальность для среднего, но независимость наблюдений и отсутствие семейной кластеризации требуют проверки по дизайну сбора. Значимость не равна клинической значимости.

## Психологические показатели

In [ ]:
psy=pd.read_excel(DATA/'Психологи.xlsx');psy.columns=psy.columns.str.strip();psy['пол']=psy['пол'].str.strip().str.lower()
display(psy.head());display(psy['пол'].value_counts());display(psy.isna().sum())
psy_cols=[c for c in psy.select_dtypes('number').columns if c not in ['учащиеся','профиль ВУЗа']]
psy_tests=compare_groups(psy,'пол',psy_cols,'м','ж');display(psy_tests.sort_values('p'))
normal_psy=[]
for c in psy_cols:
    for g in ['м','ж']:
        x=psy.loc[psy['пол'].eq(g),c].dropna();normal_psy.append([c,g,len(x),stats.shapiro(x).pvalue])
display(pd.DataFrame(normal_psy,columns=['шкала','пол','n','p_Shapiro']))
psy_tests.to_csv('psychology_welch.csv',index=False)

Минимальное p у шкалы «экстраверсия- интроверсия», максимальное — «средний IQ». Это сила статистического свидетельства, не размер различия: величина эффекта отдельно дана в Cohen d. После Holm значимых шкал 1 из 15. При малой выборке, ненормальности или порядковых шкалах выводы t-тестов требуют осторожности; группы могут различаться также по классу и профилю, поэтому причинный эффект пола не установлен.

## Бонус Iris Функция Gauss
H0: признак имеет нормальное распределение с оценёнными μ и σ; H1: распределение отличается. Критерий χ² Пирсона строится по равновероятным интервалам нормального распределения с крайними границами ±∞. Ожидаемая частота не меньше 5. Два параметра оценены по выборке, поэтому ddof=2 и число степеней свободы k−3. Применимость асимптотического приближения ограничена, особенно при малых выборках и округлении измерений.

In [ ]:
iris_raw=pd.read_excel(DATA/'Iris.xlsx',sheet_name='Data')
iris_raw['Species_name']=iris_raw['Species_name'].str.strip()
translations={'Petal_width':'Ширина лепестка','Petal_length':'Длина лепестка','Sepal_width':'Ширина чашелистика','Sepal_length':'Длина чашелистика'}
iris=iris_raw.drop(columns=['Species_No','Species_name']).rename(columns=translations);iris.index=np.arange(1,151);display(iris.head())
def Gauss(x,name):
    x=np.asarray(pd.Series(x).dropna(),dtype=float);n=len(x);k=min(8,n//10)
    mu=x.mean();sigma=x.std(ddof=0)
    if k<4 or sigma<=0:raise ValueError('Недостаточно данных для χ² с двумя оценёнными параметрами')
    edges=stats.norm.ppf(np.linspace(0,1,k+1),loc=mu,scale=sigma)
    observed,_=np.histogram(x,bins=edges);expected=np.full(k,n/k)
    chi,p=stats.chisquare(observed,expected,ddof=2)
    print(name,'n=',n,'mean=',mu,'median=',np.median(x),'skew=',stats.skew(x,bias=False),'excess=',stats.kurtosis(x,bias=False))
    print('H0: нормальность; H1: не нормальность; χ²=',chi,'df=',k-3,'p=',p,'; H0', 'отклонена' if p<.05 else 'не отклонена')
    sns.histplot(x,kde=True);plt.title(name);plt.xlabel('Сантиметры');plt.ylabel('Частота');show('iris_'+str(Gauss.counter));Gauss.counter+=1
    return p
Gauss.counter=0
iris_p={c:Gauss(iris[c],c) for c in iris.columns};display(pd.Series(iris_p,name='p χ²'))
worst=min(iris_p,key=iris_p.get);original=next(k for k,v in translations.items() if v==worst)
for species,part in iris_raw.groupby('Species_name'):Gauss(part[original],worst+' '+species)
Gauss(iris_raw.loc[iris_raw.Species_name.eq('Setosa'),original],worst+' мелкие Setosa')
Gauss(iris_raw.loc[~iris_raw.Species_name.eq('Setosa'),original],worst+' крупные два сорта')

## Общий вывод
Смесь сортов может быть ненормальной даже при близких к нормальным распределениях отдельных сортов. Функция проверяет каждый сорт и объединение двух крупных сортов отдельно; результаты приведены непосредственно над выводом. Неотклонение нормальности не означает её доказанности. Во всех сравнениях нужно учитывать размер эффекта, выбор метода и множественность проверок, а не только порог 0,05.